# Fine-tuning BioBERT sur PubMedQA

## 📋 Objectif
Adapter un modèle de Question-Réponse médicale (BioBERT) sur le dataset PubMedQA.

## 📊 Structure des données
- **PQA-A (Artificiel)** : 211 300 instances → Phase 1 d'adaptation
- **PQA-L (Étiqueté/Expert)** : 1 000 instances → Phase 2 de fine-tuning final
- **PQA-U (Non étiqueté)** : 61 200 instances → Optionnel pour pseudo-labeling

## 🎯 Stratégie d'entraînement
1. Fine-tune sur PQA-A (211k) pour adaptation au domaine
2. Fine-tune supervisé sur 80% de PQA-L
3. Évaluation sur 20% de PQA-L

**Performances attendues** : ~68-73% accuracy (SOTA) | Médecin qualifié : 78%

In [ ]:
# ============================================
# 1. INSTALLATION DES DÉPENDANCES
# ============================================

!pip install -q transformers datasets torch accelerate scikit-learn matplotlib seaborn tqdm

In [ ]:
# ============================================
# 2. IMPORTATION DES BIBLIOTHÈQUES
# ============================================

import os
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import load_dataset, Dataset as HFDataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42

# Reproductibilité
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print(f"✅ Device: {DEVICE}")
print(f"✅ CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================
# 3. CHARGEMENT DES DONNÉES PUBMEDQA
# ============================================

def load_pubmedqa_data():
    """
    Charge les trois sous-ensembles de PubMedQA depuis 'qiaojin/PubMedQA'.
    Chaque sous-ensemble est une configuration distincte.
    """
    print("📥 Chargement du dataset PubMedQA (format Parquet)...")

    # Chargement des trois configurations séparément
    pqa_artificial = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")
    pqa_labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    pqa_unlabeled = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")

    # Regroupement dans un dictionnaire pour garder la même interface
    dataset = {
        'pqa_artificial': pqa_artificial,
        'pqa_labeled': pqa_labeled,
        'pqa_unlabeled': pqa_unlabeled
    }

    print(f"✅ Datasets chargés !")
    print(f"   - PQA-A (artificiel): {len(dataset['pqa_artificial'])} instances")
    print(f"   - PQA-L (étiqueté/expert): {len(dataset['pqa_labeled'])} instances")
    print(f"   - PQA-U (non étiqueté): {len(dataset['pqa_unlabeled'])} instances")

    return dataset

# Chargement
dataset = load_pubmedqa_data()

# Affichage d'un exemple
print("\n📝 Exemple d'instance PQA-L (expert) :")
example = dataset['pqa_labeled'][0]

# Adaptation du nom des champs
answer = example['final_decision']
question = example['question']
context = example['context']

print(f"   Question: {question}")
print(f"   Contexte: {context['contexts'][0][:200]}...")
print(f"   Réponse: {answer}")  # 'yes', 'no', ou 'maybe'

In [ ]:
# ============================================
# 4. PRÉPARATION DES DONNÉES (VERSION CORRIGÉE)
# ============================================

def preprocess_pubmedqa_example(example, tokenizer, max_length=512):
    """
    Prépare un exemple pour le modèle de classification.
    Format: [CLS] Question [SEP] Contexte [SEP]
    """
    question = example['question']

    # Le contexte a une structure différente : {'contexts': ['phrase1', 'phrase2', ...]}
    if 'context' in example and isinstance(example['context'], dict):
        context = ' '.join(example['context']['contexts'])
    elif isinstance(example['context'], list):
        context = ' '.join(example['context'])
    else:
        context = str(example['context'])

    # Tokenisation
    encoding = tokenizer(
        question,
        context,
        max_length=max_length,
        padding='max_length',
        truncation='only_second',
        return_tensors='pt'
    )

    result = {
        'input_ids': encoding['input_ids'].squeeze(0),
        'attention_mask': encoding['attention_mask'].squeeze(0),
    }

    # Ajout du label si disponible (champ 'final_decision' au lieu de 'answer')
    if 'final_decision' in example and example['final_decision']:
        label_map = {'yes': 0, 'no': 1, 'maybe': 2}
        answer = example['final_decision']
        result['label'] = label_map.get(answer, -1)

    return result


def prepare_datasets(dataset_dict, tokenizer, sample_size_artificial=None):
    """
    Prépare les datasets pour l'entraînement.
    """
    print("🔄 Prétraitement des données...")

    # 1. Dataset artificiel (PQA-A)
    pqa_artificial = dataset_dict['pqa_artificial']
    if sample_size_artificial and sample_size_artificial < len(pqa_artificial):
        indices = random.sample(range(len(pqa_artificial)), sample_size_artificial)
        pqa_artificial = pqa_artificial.select(indices)

    print(f"   - Traitement PQA-A ({len(pqa_artificial)} instances)...")
    pqa_artificial_processed = pqa_artificial.map(
        lambda x: preprocess_pubmedqa_example(x, tokenizer),
        remove_columns=pqa_artificial.column_names
    )

    # 2. Dataset étiqueté (PQA-L)
    pqa_labeled = dataset_dict['pqa_labeled']
    print(f"   - Traitement PQA-L ({len(pqa_labeled)} instances)...")
    pqa_labeled_processed = pqa_labeled.map(
        lambda x: preprocess_pubmedqa_example(x, tokenizer),
        remove_columns=pqa_labeled.column_names
    )

    # Split train/test pour PQA-L (80/20)
    pqa_labeled_split = pqa_labeled_processed.train_test_split(test_size=0.2, seed=SEED)

    print(f"\n✅ Prétraitement terminé !")
    print(f"   - PQA-A (adaptation): {len(pqa_artificial_processed)} instances")
    print(f"   - PQA-L train: {len(pqa_labeled_split['train'])} instances")
    print(f"   - PQA-L test: {len(pqa_labeled_split['test'])} instances")

    return {
        'artificial': pqa_artificial_processed,
        'labeled_train': pqa_labeled_split['train'],
        'labeled_test': pqa_labeled_split['test']
    }


# Initialisation du tokenizer BioBERT
print("🔤 Chargement du tokenizer BioBERT...")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

# Préparation des datasets
prepared_datasets = prepare_datasets(
    dataset,
    tokenizer,
    sample_size_artificial=10000  # Réduire pour test rapide, None pour tout
)

In [ ]:
# ============================================
# 5. FONCTIONS D'ÉVALUATION ET MÉTRIQUES
# ============================================

def compute_metrics(eval_pred):
    """
    Calcule les métriques d'évaluation pour la classification.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


def plot_confusion_matrix(y_true, y_pred, title="Matrice de confusion"):
    """
    Affiche la matrice de confusion.
    """
    cm = confusion_matrix(y_true, y_pred)
    labels = ['Yes', 'No', 'Maybe']

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel('Prédictions')
    plt.ylabel('Vraies valeurs')
    plt.tight_layout()
    plt.show()


def plot_training_history(trainer_state, title="Historique d'entraînement"):
    """
    Affiche les courbes de loss et d'accuracy.
    """
    history = pd.DataFrame(trainer_state.log_history)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    loss_data = history.dropna(subset=['loss'])
    eval_loss_data = history.dropna(subset=['eval_loss'])

    axes[0].plot(loss_data['step'], loss_data['loss'], label='Train Loss', marker='o')
    if len(eval_loss_data) > 0:
        axes[0].plot(eval_loss_data['step'], eval_loss_data['eval_loss'],
                     label='Eval Loss', marker='s')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Évolution de la Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    eval_acc_data = history.dropna(subset=['eval_accuracy'])
    if len(eval_acc_data) > 0:
        axes[1].plot(eval_acc_data['step'], eval_acc_data['eval_accuracy'],
                     label='Eval Accuracy', marker='s', color='green')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('Évolution de l\'Accuracy')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================
# 6. PHASE 1: FINE-TUNING SUR PQA-A (ARTIFICIEL)
# ============================================

def train_phase1_adaptation(dataset_artificial, output_dir="./phase1_biobert_artificial"):
    """
    Phase 1: Adaptation du modèle BioBERT sur les données artificielles PQA-A.
    Cette phase permet d'apprendre les motifs linguistiques du domaine médical.
    """
    print("=" * 60)
    print("🚀 PHASE 1: Fine-tuning sur PQA-A (données artificielles)")
    print("=" * 60)

    # Chargement du modèle BioBERT
    print("\n📦 Chargement de BioBERT...")
    model = AutoModelForSequenceClassification.from_pretrained(
        "dmis-lab/biobert-base-cased-v1.1",
        num_labels=3,  # yes, no, maybe
        id2label={0: 'yes', 1: 'no', 2: 'maybe'},
        label2id={'yes': 0, 'no': 1, 'maybe': 2}
    )
    model.to(DEVICE)

    # Split du dataset artificiel pour avoir un set de validation
    split_dataset = dataset_artificial.train_test_split(test_size=0.1, seed=SEED)
    train_dataset = split_dataset['train']
    eval_dataset = split_dataset['test']

    print(f"   - Train: {len(train_dataset)} instances")
    print(f"   - Eval: {len(eval_dataset)} instances")

    # Configuration de l'entraînement
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,  # 3 epochs suffisent pour l'adaptation
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_ratio=0.1,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_dir=f'{output_dir}/logs',
        logging_steps=100,
        eval_strategy='steps',
        eval_steps=500,
        save_strategy='steps',
        save_steps=1000,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        report_to='none',
        seed=SEED
    )

    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # Entraînement
    print("\n🔄 Entraînement en cours...")
    trainer.train()

    # Évaluation
    print("\n📊 Évaluation finale sur le set de validation artificiel:")
    eval_results = trainer.evaluate()
    print(f"   Accuracy: {eval_results.get('eval_accuracy', 0):.4f}")
    print(f"   F1-score: {eval_results.get('eval_f1', 0):.4f}")

    # Sauvegarde du modèle
    trainer.save_model(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")

    print(f"\n✅ Phase 1 terminée ! Modèle sauvegardé dans {output_dir}/final")

    return trainer, model

In [ ]:
# Exécution de la Phase 1
# Note: Cette phase peut prendre plusieurs heures sur les 211k instances complètes
# Avec 10k instances, cela prend environ 15-30 minutes sur GPU T4

phase1_trainer, phase1_model = train_phase1_adaptation(
    dataset_artificial=prepared_datasets['artificial'],
    output_dir="./phase1_biobert_artificial"
)

# Affichage de l'historique
if phase1_trainer:
    plot_training_history(phase1_trainer.state, title="Phase 1 - Adaptation sur PQA-A")

In [ ]:
# ============================================
# 7. PHASE 2: FINE-TUNING FINAL SUR PQA-L (EXPERT)
# ============================================

def train_phase2_final(base_model, dataset_train, dataset_test, output_dir="./phase2_biobert_expert"):
    """
    Phase 2: Fine-tuning final sur les données expertes PQA-L.
    Cette phase affine le modèle sur les annotations de haute qualité.
    """
    print("=" * 60)
    print("🚀 PHASE 2: Fine-tuning final sur PQA-L (données expertes)")
    print("=" * 60)

    print(f"   - Train: {len(dataset_train)} instances")
    print(f"   - Test: {len(dataset_test)} instances")

    # Configuration de l'entraînement - plus prudente car peu de données
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,  # Plus d'epochs car dataset petit
        per_device_train_batch_size=8,  # Batch plus petit
        per_device_eval_batch_size=16,
        warmup_ratio=0.1,
        learning_rate=1e-5,  # Learning rate plus faible pour ne pas oublier
        weight_decay=0.01,
        logging_dir=f'{output_dir}/logs',
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        report_to='none',
        seed=SEED
    )

    # Data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Trainer
    trainer = Trainer(
        model=base_model,
        args=training_args,
        train_dataset=dataset_train,
        eval_dataset=dataset_test,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )

    # Entraînement
    print("\n🔄 Entraînement en cours...")
    trainer.train()

    # Sauvegarde du modèle final
    trainer.save_model(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")

    print(f"\n✅ Phase 2 terminée ! Modèle final sauvegardé dans {output_dir}/final")

    return trainer

In [ ]:
# Exécution de la Phase 2
# On reprend le modèle de la phase 1

phase2_trainer = train_phase2_final(
    base_model=phase1_model,
    dataset_train=prepared_datasets['labeled_train'],
    dataset_test=prepared_datasets['labeled_test'],
    output_dir="./phase2_biobert_expert"
)

# Affichage de l'historique
if phase2_trainer:
    plot_training_history(phase2_trainer.state, title="Phase 2 - Fine-tuning sur PQA-L")

In [ ]:
# ============================================
# 8. ÉVALUATION FINALE DÉTAILLÉE
# ============================================

def evaluate_final_model(trainer, test_dataset):
    """
    Évaluation complète du modèle final.
    """
    print("=" * 60)
    print("📊 ÉVALUATION FINALE DU MODÈLE")
    print("=" * 60)

    # Prédictions
    predictions = trainer.predict(test_dataset)
    preds = np.argmax(predictions.predictions, axis=1)
    labels = predictions.label_ids

    # Métriques globales
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')

    print(f"\n📈 Métriques globales:")
    print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-score:  {f1:.4f}")

    # Comparaison avec les benchmarks
    print(f"\n📊 Comparaison avec l'état de l'art:")
    print(f"   Notre modèle:     {accuracy*100:.1f}%")
    print(f"   SOTA BioBERT:     ~68-73%")
    print(f"   Médecin qualifié: ~78%")

    # Rapport par classe
    print(f"\n📋 Rapport détaillé par classe:")
    label_names = ['yes', 'no', 'maybe']
    print(classification_report(labels, preds, target_names=label_names))

    # Matrice de confusion
    plot_confusion_matrix(labels, preds, "Matrice de confusion - Modèle final")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': preds,
        'labels': labels
    }

# Évaluation finale
final_results = evaluate_final_model(phase2_trainer, prepared_datasets['labeled_test'])

In [ ]:
# ============================================
# 9. ANALYSE DES ERREURS
# ============================================

def analyze_errors(dataset_test, predictions, labels, num_examples=5):
    """
    Analyse les erreurs de prédiction du modèle.
    """
    print("=" * 60)
    print("🔍 ANALYSE DES ERREURS")
    print("=" * 60)

    label_names = ['yes', 'no', 'maybe']

    # Trouver les erreurs
    errors = []
    for i, (true_label, pred_label) in enumerate(zip(labels, predictions)):
        if true_label != pred_label:
            errors.append({
                'index': i,
                'true': label_names[true_label],
                'pred': label_names[pred_label]
            })

    print(f"\n📊 Statistiques d'erreurs:")
    print(f"   Total erreurs: {len(errors)} sur {len(labels)} ({len(errors)/len(labels)*100:.1f}%)")

    # Distribution des types d'erreurs
    error_types = {}
    for err in errors:
        key = f"{err['true']} → {err['pred']}"
        error_types[key] = error_types.get(key, 0) + 1

    print(f"\n   Distribution des erreurs:")
    for err_type, count in sorted(error_types.items(), key=lambda x: -x[1]):
        print(f"      {err_type}: {count}")

    # Afficher quelques exemples d'erreurs
    print(f"\n📝 Exemples d'erreurs:")
    original_dataset = dataset['pqa_labeled']

    for i, err in enumerate(errors[:num_examples]):
        idx = err['index']
        example = original_dataset[idx]

        print(f"\n   --- Erreur {i+1} ---")
        print(f"   Question: {example['question'][:200]}...")
        print(f"   Vraie réponse: {err['true']}")
        print(f"   Prédiction:    {err['pred']}")

# Analyse des erreurs
analyze_errors(
    prepared_datasets['labeled_test'],
    final_results['predictions'],
    final_results['labels']
)

In [ ]:
# ============================================
# 10. INFÉRENCE SUR DE NOUVELLES QUESTIONS
# ============================================

class PubMedQAPredictor:
    """
    Classe pour faire des prédictions sur de nouvelles questions médicales.
    """

    def __init__(self, model_path="./phase2_biobert_expert/final"):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.to(self.device)
        self.model.eval()
        self.label_names = ['yes', 'no', 'maybe']

    def predict(self, question, context, max_length=512):
        """
        Prédit la réponse à une question médicale.
        """
        # Tokenisation
        encoding = self.tokenizer(
            question,
            context,
            max_length=max_length,
            padding='max_length',
            truncation='only_second',
            return_tensors='pt'
        )

        # Déplacement sur le device
        input_ids = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)

        # Prédiction
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=-1)
            prediction = torch.argmax(logits, dim=-1)

        probs = probabilities[0].cpu().numpy()
        pred_label = self.label_names[prediction[0].item()]

        return {
            'prediction': pred_label,
            'confidence': float(probs.max()),
            'probabilities': {
                'yes': float(probs[0]),
                'no': float(probs[1]),
                'maybe': float(probs[2])
            }
        }


# Initialisation du prédicteur
predictor = PubMedQAPredictor()

# Test sur un exemple du dataset
print("=" * 60)
print("🧪 TEST D'INFÉRENCE")
print("=" * 60)

test_example = dataset['pqa_labeled'][0]
question = test_example['question']

if isinstance(test_example['context'], dict):
    context = ' '.join(test_example['context']['contexts'])
elif isinstance(test_example['context'], list):
    context = ' '.join(test_example['context'])
else:
    context = test_example['context']

true_answer = test_example['final_decision']

print(f"\n📝 Question: {question}")
print(f"📄 Contexte (début): {context[:200]}...")
print(f"✅ Réponse attendue: {true_answer}")

result = predictor.predict(question, context)
print(f"\n🤖 Prédiction du modèle: {result['prediction']}")
print(f"📊 Confiance: {result['confidence']:.3f}")
print(f"📈 Probabilités: yes={result['probabilities']['yes']:.3f}, "
      f"no={result['probabilities']['no']:.3f}, maybe={result['probabilities']['maybe']:.3f}")

In [ ]:
# ============================================
# 11. SAUVEGARDE ET EXPORT DU MODÈLE
# ============================================

def save_model_for_deployment(model_path="./phase2_biobert_expert/final", export_dir="./pubmedqa_biobert_final"):
    """
    Sauvegarde le modèle dans un format facile à déployer.
    """
    import shutil

    print("💾 Sauvegarde du modèle pour déploiement...")

    # Création du répertoire d'export
    os.makedirs(export_dir, exist_ok=True)

    # Copie des fichiers du modèle
    for file in os.listdir(model_path):
        src = os.path.join(model_path, file)
        dst = os.path.join(export_dir, file)
        if os.path.isfile(src):
            shutil.copy2(src, dst)

    # Création d'un fichier de métadonnées
    metadata = {
        'model_name': 'BioBERT fine-tuned on PubMedQA',
        'base_model': 'dmis-lab/biobert-base-cased-v1.1',
        'training_data': 'PubMedQA (PQA-A: 211k, PQA-L: 1k)',
        'labels': ['yes', 'no', 'maybe'],
        'max_length': 512,
        'performance': {
            'accuracy': float(final_results['accuracy']),
            'f1_score': float(final_results['f1'])
        }
    }

    with open(os.path.join(export_dir, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"✅ Modèle exporté dans {export_dir}/")
    print(f"   Fichiers sauvegardés:")
    for file in os.listdir(export_dir):
        print(f"      - {file}")

# Sauvegarde finale
save_model_for_deployment()

In [ ]:
# ============================================
# 12. RÉSUMÉ ET CONCLUSION
# ============================================

print("=" * 60)
print("📋 RÉSUMÉ DE L'ENTRAÎNEMENT")
print("=" * 60)

print(f"""
✅ PIPELINE COMPLÉTÉE AVEC SUCCÈS

1. 📥 Données chargées:
   - PQA-A (artificiel): 211,300 instances
   - PQA-L (expert): 1,000 instances

2. 🔄 Phase 1 - Adaptation sur PQA-A:
   - Modèle: BioBERT base
   - Objectif: Apprentissage des motifs linguistiques médicaux

3. 🎯 Phase 2 - Fine-tuning sur PQA-L:
   - Split: 80% train, 20% test
   - Objectif: Optimisation sur annotations expertes

4. 📊 Performances finales:
   - Accuracy: {final_results['accuracy']:.2%}
   - F1-score: {final_results['f1']:.3f}
   - Benchmark SOTA: ~68-73%

5. 💾 Modèle sauvegardé:
   - ./pubmedqa_biobert_final/

🚀 UTILISATION FUTURE:
   predictor = PubMedQAPredictor('./pubmedqa_biobert_final')
   result = predictor.predict(question, context)
""")

# Visualisation finale des performances
fig, ax = plt.subplots(figsize=(10, 6))

benchmarks = {
    'Notre modèle': final_results['accuracy'] * 100,
    'SOTA BioBERT': 68,
    'Médecin qualifié': 78.0
}

colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(benchmarks.keys(), benchmarks.values(), color=colors, alpha=0.8)

ax.set_ylabel('Accuracy (%)')
ax.set_title('Comparaison des performances sur PubMedQA')
ax.set_ylim(0, 100)
ax.axhline(y=78, color='#e74c3c', linestyle='--', alpha=0.5, label='Performance médecin')

for bar, value in zip(bars, benchmarks.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{value:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.legend()
plt.tight_layout()
plt.show()

print("\n✨ Notebook terminé avec succès !")


In [ ]:
# Pour télécharger le modèle
import shutil
import os
from google.colab import files

# Créer l'archive
model_path = "./pubmedqa_biobert_final"
shutil.make_archive('pubmedqa_model', 'zip', model_path)

# Télécharger
files.download('pubmedqa_model.zip')